# 01: Exploring raw OPM-MEG files

**Who this is for:** new users of the Scully Center MEG lab.

**What you will do:**

1. Put your raw data in place (or copy the sample dataset from the file server).
2. Convert the raw `.cMEG` recordings to FIF with **cMEG2FIF**, in your Terminal.
3. Explore the files, compare what Cerca writes with what the converter produces.

## 0. Setup

Choose **one** of the two setups, then set your paths in 0.3:

- **0.1 Option A: Cluster (preferred).** Jupyter runs on scotty and you use it from your laptop's browser. The data never leave the server.
- **0.2 Option B: Local laptop.** Jupyter runs on your laptop, reading the data from your mounted lab partition.

### 0.1 Option A: Cluster setup (preferred)

#### 0.1.1 Pick a port and SSH in with the tunnel:

Choose something in the 4000-9999 range that avoids 8800-8899. On a Mac, also avoid 5000 and 7000.
In your Terminal ssh to scotty:

```ssh -L 4817:localhost:4817 NETID@scotty.pni.princeton.edu```

Check that nobody else is using the port on scotty:
```ss -ltn | grep 4817```


#### 0.1.2 Create the conda environment

Create an environment from scratch:

```bash
conda create -n pygers-meg -c conda-forge python=3.11 mne jupyterlab pandas ipykernel ipympl mne-bids
conda activate pygers-meg
pip install "cmeg2fif[plot] @ git+https://github.com/mrribbits/cMEG2FIF-Scully.git"
pip install bids-validator-deno
```

Or create it using the supplied scotty-environment.yml:
```bash
conda env create -f scotty-environment.yml -n pygers-meg
conda activate pygers-meg
```

Check the converter installed:

```bash
cmeg2fif --version
```

#### 0.1.3 Launch Jupyter on scotty
```
jupyter lab --no-browser --ip=127.0.0.1 --port=4817 --ServerApp.port_retries=0
```

#### 0.1.4 Open on your laptop's browser
Copy the URL that Jupyter prints, which looks like `http://127.0.0.1:4817/lab?token=...`, and paste it into your local browser.

#### 0.1.5 To shut down
Press Ctrl-C twice in the scotty session. If you ever leave one running by accident, `jupyter server list` shows it and `jupyter server stop 4817` stops it.


### 0.2 Option B: Local laptop setup

#### 0.2.1 Python environment (do this once, in your Terminal)

Create an environment:

```bash
conda create -n pygers-meg -c conda-forge python=3.11 mne jupyterlab pandas ipympl mne-bids
conda activate pygers-meg
pip install "cmeg2fif[plot] @ git+https://github.com/mrribbits/cMEG2FIF-Scully.git"
pip install bids-validator-deno
```

Check the converter installed:

```bash
cmeg2fif --version
```

#### 0.2.2 Mount your 'cup' lab partition

All data stay on the server: **do not copy them to your laptop.** Mount the server in Finder (macOS: *Go → Connect to Server…*) or File Explorer (Windows: *Map network drive*).


### Running this notebook from the pipeline

**Run the two cells below first, also when you run the notebook yourself.** They define `from_config()`,
which the settings cells use. Run by hand, `config_file` stays `None` and they change nothing: the notebook
uses the settings you edit in its cells. They let the Pygers-MEG-to-BIDS pipeline (`run_bids-pipeline.sh`) run this notebook
with the settings from a study's YAML files instead.

In pipeline mode, each settings cell ends with a line like
`EVENT_ID = from_config("EVENT_ID")`, which replaces the value typed above it with the one from the
YAML files. Run by hand, that line leaves the value as you typed it.

In [ ]:
# Left as None when you run the notebook yourself. The pipeline replaces it with the path of its
# settings file (papermill adds a cell right below this one that sets it).
config_file = None

In [ ]:
# Nothing to edit here. In pipeline mode (config_file set above), this reads this notebook's settings
# from the pipeline's settings file. Run by hand, PIPELINE stays None and from_config() changes nothing.
from pathlib import Path   # file paths that work on macOS, Linux and Windows

NOTEBOOK_ID = "01"     # this notebook's section in the pipeline's settings file
PIPELINE = None            # None = running by hand
if config_file is not None:
    import yaml                                       # reads .yaml files
    with open(config_file, encoding="utf-8") as f:
        PIPELINE = yaml.safe_load(f)[NOTEBOOK_ID]     # a dict: setting name -> value
    print(f"Pipeline mode: notebook {NOTEBOOK_ID} settings from {config_file}")


def from_config(*names):
    """The named settings: from the pipeline's settings file in pipeline mode, unchanged otherwise.

    Example:  EVENT_ID, DROP_UNMAPPED = from_config("EVENT_ID", "DROP_UNMAPPED")

    In pipeline mode every name must be in the settings file: a missing one stops the notebook, so a
    setting can never silently keep this notebook's example value. Settings the file lists under
    "_paths" are turned into Paths.
    """
    if PIPELINE is None:
        values = [globals()[name] for name in names]          # run by hand: keep the values typed above
    else:
        missing = [name for name in names if name not in PIPELINE]
        if missing:
            raise KeyError(f"The pipeline settings file has no value for {missing} (notebook {NOTEBOOK_ID}).")
        paths = set(PIPELINE.get("_paths", []))
        values = [Path(PIPELINE[name]) if name in paths and PIPELINE[name] is not None else PIPELINE[name]
                  for name in names]
    return values[0] if len(values) == 1 else tuple(values)

### 0.3 Your raw data and paths

#### 0.3.1 Where the raw data go

Each recording session has its own folder in your project's `sourcedata/`, and the files from the lab go,
unchanged, in its `raw/` folder. Set `RAW_DIR` in the cell below to that `raw/` folder (replace
`[yourpath]` with a folder in your lab's partition; `auditory-oddball` is the sample project):

```
<project>/
├── bids/                          ← the BIDS dataset (notebooks 04-06 write it)
└── sourcedata/
    └── sub-01/
        └── ses-01/
            ├── raw/               ← RAW_DIR: the files from the lab, as they came (never changed)
            │   ├── cMEG/          ← one <prefix>_cMEG_Data folder per recording (task + empty room)
            │   ├── einscan/       ← head digitization (…_dev2head_xfm.tsv, …_xfmd.xyz)
            │   ├── eye/           ← the EyeLink .edf (notebook 02)
            │   ├── mri/           ← the DICOM files, directly or in a dcm/ subfolder (notebook 03)
            │   └── behav/         ← the PsychoPy .csv logs (notebook 04)
            ├── fif/               ← this notebook's FIF files (OUT_DIR, 0.3.2)
            ├── eye-staging/       ← notebook 02's eye2bids output
            ├── mri-staging/       ← notebook 03's HeuDiConv output
            ├── defaced_anat/      ← notebook 03's defaced MRI copies (for sharing)
            └── reports/           ← the pipeline's HTML reports
```

The other notebooks write their working files next to `raw/` in the same session folder, so `raw/` only
ever holds what came from the lab. If a session has no eye tracking, MRI or behavioral logs, leave that
folder out.

**To use the sample dataset**, uncomment the two lines marked in the cell, run it once, and put the `#`
back. They copy the shared sample `raw/` folder into your `RAW_DIR` (the shared folder is never changed).
This works as written when Jupyter runs on scotty (Option A). On a laptop (Option B), `RAW_DIR` must be as
your laptop sees it through the mounted share (`/Volumes/...` on a Mac), and the `!` commands need macOS
or Linux.

The cell then checks that `RAW_DIR` exists and lists which of the five folders it found.

In [ ]:
from pathlib import Path

# ---- EDIT THIS LINE ------------------------------------------------------
# Your session's raw folder: <project>/sourcedata/sub-NN/ses-NN/raw (see the layout above)
RAW_DIR = Path("/jukebox/[yourpath]/auditory-oddball/sourcedata/sub-01/ses-01/raw")
# ---------------------------------------------------------------------------

# Pipeline mode: the session's raw folder from the YAML files (run by hand: unchanged)
RAW_DIR = from_config("RAW_DIR")

# ---- Sample dataset ----------------------------------------------------------
# To work with the sample dataset, uncomment the two lines below (delete the "# " at the start of
# each), run the cell once, then put the "# " back. They create RAW_DIR and copy the shared, read-only
# sample raw folder into it. rsync copies only new or changed files, so a second run is quick.
# !mkdir -p "{RAW_DIR}"
# !rsync -av "/jukebox/scratch/mpinsk/MEG/pygers-meg/auditory-oddball/sub-01/raw/" "{RAW_DIR}/"

# ---- Check the raw folder ----------------------------------------------------
# This notebook needs cMEG/ and einscan/; the others are used by notebooks 02-04.
assert RAW_DIR.is_dir(), f"RAW_DIR not found: {RAW_DIR}"
for name, used_by in [("cMEG", "this notebook"), ("einscan", "this notebook"), ("eye", "notebook 02"),
                      ("mri", "notebook 03"), ("behav", "notebook 04")]:
    status = "found  " if (RAW_DIR / name).is_dir() else "MISSING"
    print(f"   {status}  {name + '/':9s} ({used_by})")
assert (RAW_DIR / "cMEG").is_dir(), f"No cMEG/ folder in {RAW_DIR}: see the layout above."

#### 0.3.2 Set the output folder

Edit the lines between the markers below (use `/jukebox/...` on scotty, `/Volumes/...` on a laptop), then run the cell: `OUT_DIR`, the BIDS labels (used to name the FIF files), the two recording prefixes and the EinScan files. `RAW_DIR` comes from 0.3.1.

In [ ]:
from pathlib import Path

# ---- EDIT THESE LINES ------------------------------------------------------
OUT_DIR = Path("/jukebox/[yourpath]/auditory-oddball/sourcedata/sub-01/ses-01/fif")  # converter output, next to raw/

# BIDS labels, used to name the FIF files (letters and digits only). Use the same labels in
# notebooks 02-05, so all the files of this recording line up.
SUBJECT = "01"               # -> sub-01
SESSION = "01"               # -> ses-01 (None, without quotes, if your study has no sessions)
TASK    = "oddballftcued"    # -> task-oddballftcued
RUN     = None               # -> no run- label. Several runs of the task, each its own recording: "01",
                             #    "02", ... (in quotes), and DATA_PREFIX below is that run's recording

# Recording prefixes = the acquisition start time Cerca put in the folder names (<prefix>_cMEG_Data)
ER_PREFIX   = "20260929_125813"   # empty-room (noise) recording
DATA_PREFIX = "20260929_134022"   # task recording (auditory oddball; with RUN, this run's recording)

# EinScan head-digitization files (in the raw folder's einscan/)
XFM_FILE = RAW_DIR / "einscan" / "sub-01-head+helmet_dev2head_xfm.tsv"
XYZ_FILE = RAW_DIR / "einscan" / "sub-01-head+helmet_digitisation_from_mesh_3_xfmd.xyz"

LINE_FREQ = 60                    # North American mains
# ---------------------------------------------------------------------------
# RAW_DIR was set in 0.3.1

# Pipeline mode: these come from the YAML files (run by hand: unchanged)
(OUT_DIR, SUBJECT, SESSION, TASK, RUN, ER_PREFIX, DATA_PREFIX, XFM_FILE, XYZ_FILE, LINE_FREQ) = from_config(
    "OUT_DIR", "SUBJECT", "SESSION", "TASK", "RUN", "ER_PREFIX", "DATA_PREFIX", "XFM_FILE", "XYZ_FILE", "LINE_FREQ")

ER_DIR    = RAW_DIR / "cMEG" / f"{ER_PREFIX}_cMEG_Data"
DATA_DIR  = RAW_DIR / "cMEG" / f"{DATA_PREFIX}_cMEG_Data"
EINSCAN   = RAW_DIR / "einscan"

# Converter outputs, with BIDS-style names built from the labels (so they can drop straight into a
# BIDS project later), e.g. sub-01_ses-01_task-noise_meg.fif. The empty room has no run label: every
# run of the session shares it.
stem = f"sub-{SUBJECT}" + (f"_ses-{SESSION}" if SESSION else "")
ER_FIF   = OUT_DIR / f"{stem}_task-noise_meg.fif"
DATA_FIF = OUT_DIR / (f"{stem}_task-{TASK}" + (f"_run-{RUN}" if RUN else "") + "_meg.fif")
print("Empty-room FIF:", ER_FIF)
print("Task FIF      :", DATA_FIF)

## 1. What's in the raw session? (file inventory)

Cerca's acquisition software writes these files next to each other, sharing the start-time prefix:

| File | Contents |
|---|---|
| `<prefix>_meg_NNN.cMEG` | Raw data, split into numbered parts |
| `<prefix>_meg.json` | Cerca's sidecar: sampling rate, duration, a comment (Section 2) |
| `<prefix>_channels.tsv` | Channel names, types, gains (V/nT), good/bad status |
| `<prefix>_HelmConfig.tsv` | Sensor positions and orientations in the helmet |
| `<prefix>_SensorTransform.tsv` | Cerca's default sensor transform (we use the cReg `--xfm` instead) |
| `<prefix>_SessionInfo.txt` | Session notes from the acquisition software |

The other folders: `einscan/` (coregistration outputs from cReg), `behav/` (PsychoPy logs), `eye/` (EyeLink), `mri/` (DICOMs).

## 2. The original `_meg.json` sidecar

Cerca writes a `_meg.json` next to each recording. It *looks* like a BIDS sidecar. The next cell lists every field and its value, for the task recording and the empty room side by side.

In [ ]:
import json                        # read the Cerca _meg.json sidecars
import re                          # regular expressions: pick apart file names and log lines
import platform                    # report the Python version
from collections import Counter    # count things, e.g. how many channels of each type

import numpy as np                 # arrays and maths
import pandas as pd                # tables: TSV/CSV files, summaries
import matplotlib.pyplot as plt    # plots
import mne                         # MNE-Python: reads and analyses the FIF files

# MNE prints a lot of progress messages by default. "WARNING" hides those and shows
# only warnings and errors. Use "INFO" to see everything MNE is doing.
mne.set_log_level("WARNING")

# By default pandas shortens long tables (showing "..." in the middle) and long text.
# These show up to 300 rows and up to 120 characters per cell, so whole channel lists fit.
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 120)

print(f"Python {platform.python_version()}  |  MNE {mne.__version__}  |  "
      f"NumPy {np.__version__}  |  pandas {pd.__version__}")

js_data = json.loads((DATA_DIR / f"{DATA_PREFIX}_meg.json").read_text())
js_er   = json.loads((ER_DIR   / f"{ER_PREFIX}_meg.json").read_text())

print(f"{'field':28s} {'task':>14s}   {'empty room':>14s}")
for key in js_data:
    print(f"{key:28s} {js_data[key]!r:>14}   {js_er.get(key)!r:>14}")

**What's wrong with it?** It is not valid BIDS:

- **Descriptive fields** (`InstitutionName`, `Manufacturer`, `ManufacturersModelName`, `SoftwareVersions`, `TaskName`, …) are `"n/a"`, and `TaskDescription` is empty.
- **`PowerLineFrequency`** is `0`. BIDS requires the mains frequency in Hz (60 in the US, 50 in Europe), or `"n/a"`.
- **`RecordingType`** is `"Measurement"` (task) or `"Noise"` (empty room). BIDS allows only `"continuous"`, `"epoched"` or `"discontinuous"`.
- **Channel counts** don't match the real channel layout: the reference sensors are counted in `MEGChannelCount`, and `MEGREFChannelCount` is `0` (Section 2.1).
- **`DigitizedLandmarks`** and **`DigitizedHeadPoints`** are `false`, even for the task recording, which has a head scan.

### 2.1 Where are the reference sensors?

Cerca's `channels.tsv` uses only **three** channel types: `MEGMAG`, `TRIG` and `MISC`. But the system really records **four** kinds of channels. The two **reference sensors**, 3N and 3O, are incorrectly labelled `MEGMAG` just like the on-head sensors.

Cerca's _meg.json follows the same view. It does have a `MEGREFChannelCount` field, but sets it to `0` and counts the reference channels inside `MEGChannelCount` (198 instead of 192).

Nothing in `channels.tsv` marks a sensor as a reference. The clue is in `HelmConfig.tsv`, which lists only the sensors that sit in a **helmet slot**. A sensor that is recorded but has no slot must be a reference. The next cell makes that comparison, which is exactly what cMEG2FIF does during conversion when it labels these channels `ref_meg` in the FIF.

In [ ]:
def read_tsv(path):
    # Tab-separated file -> table. Everything kept as text; "n/a" stays "n/a"; "#" lines are comments.
    return pd.read_csv(path, sep="\t", comment="#", dtype=str, keep_default_na=False)

ch_tsv = read_tsv(DATA_DIR / f"{DATA_PREFIX}_channels.tsv")
helm   = read_tsv(DATA_DIR / f"{DATA_PREFIX}_HelmConfig.tsv")

print("channels.tsv channel types/counts:")
for ch_type, n in ch_tsv["type"].value_counts().items():
    print(f"  {ch_type:22s} {n}")

print("\n_meg.json channel types/counts:")
for key, val in js_data.items():
    if key.endswith("ChannelCount"):
        print(f"  {key:22s} {val}")

print(f"\nHelmConfig.tsv: {len(helm)} sensors in helmet slots")

## 3. Convert the recordings with cMEG2FIF

The converter reads every numbered part (`_meg_001.cMEG`, `_meg_002.cMEG`, …), joins them, attaches sensor geometry, decodes triggers and buttons, and writes a FIF plus a conversion log. See the [cMEG2FIF README](https://github.com/mrribbits/cMEG2FIF-Scully) for the full option list.

Two conversions:

- **Empty room** (`task-noise`): no head was in the helmet, so **no** `--xfm` / `--dig`.
- **Task recording** (`task-oddballftcued`): with `--xfm` (device→head transform) and `--dig` (head-shape points and fiducials) from the Einscan / cReg coregistration.

The next cell runs both conversions right here in the notebook. Before each one it prints the exact command being run, then shows the converter's output as it goes. The task recording takes several minutes.

- `--no-plot` is always included: the converter's pop-up 3D window can't open from inside a notebook.
- The converter never overwrites existing output. If a FIF is already there, the cell skips that recording. To convert again, set `FORCE = True` (this adds `--force`). If you reconvert after Section 4 has opened the FIF files, run Section 4's loading cell again so MNE reads the new files.

In [ ]:
import shlex, shutil, subprocess, sys

FORCE = False   # True = convert again even if the FIF already exists (adds --force)
FORCE = from_config("FORCE")   # pipeline mode: force_reconvert from the YAML files

OUT_DIR.mkdir(parents=True, exist_ok=True)

# The cmeg2fif command and peripherals file installed in this conda environment
CMEG2FIF = shutil.which("cmeg2fif") or str(Path(sys.executable).parent / "cmeg2fif")
PERIPHERALS = Path(sys.prefix) / "share" / "cmeg2fif" / "cMEG_peripherals.tsv"

conversions = [
    ("Empty-room recording", ER_FIF,
     [CMEG2FIF, ER_DIR / f"{ER_PREFIX}_meg_001.cMEG", "--out", ER_FIF,
      "--peripherals", PERIPHERALS, "--line-freq", LINE_FREQ, "--no-plot"]),
    ("Task recording (with head coregistration)", DATA_FIF,
     [CMEG2FIF, DATA_DIR / f"{DATA_PREFIX}_meg_001.cMEG", "--out", DATA_FIF,
      "--peripherals", PERIPHERALS, "--line-freq", LINE_FREQ,
      "--xfm", XFM_FILE, "--dig", XYZ_FILE, "--no-plot"]),
]

for label, fif, cmd in conversions:
    cmd = [str(x) for x in cmd] + (["--force"] if FORCE else [])
    print(f"===== {label}")
    print("$ " + " ".join(shlex.quote(x) for x in cmd) + "\n")     # the exact command
    if fif.exists() and not FORCE:
        print(f"{fif.name} already exists, skipping (set FORCE = True to convert again)\n")
        continue
    # Run the converter and print its output line by line as it works
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"cmeg2fif failed for {label} (exit code {proc.returncode})")
    print()

## 4. Does the FIF hold the correct values for BIDS?

**Important:** cMEG2FIF does **not** write a `_meg.json`. Its output is a FIF file (plus the log). The sidecar is written later, by the **BIDS conversion step**, which reads its values *from the FIF*.

So the question is really: does the FIF hold the correct values, so that the BIDS step will write a valid sidecar? Load the FIF files (only the headers are read, not the data), look at everything the header holds, then pick out the fields the BIDS step will use.

In [ ]:
raw    = mne.io.read_raw_fif(DATA_FIF, preload=False)
raw_er = mne.io.read_raw_fif(ER_FIF, preload=False)
print(raw)
print(raw_er)

**What's in a FIF header?** `raw.info` holds everything stored about the recording apart from the data itself: sampling rate, mains frequency, recording date, channel list, digitisation points, device-to-head transform, bad channels, and more. Displaying it shows every field that has a value:

In [ ]:
# The FIF header: every non-empty field and its value
raw.info

Some fields are empty for this recording and so don't appear above. To see every field name a FIF header can hold:

In [ ]:
# All field names in the header, including empty ones
print(sorted(raw.info.keys()))

Now pick out the fields that correspond to the problems in Cerca's `_meg.json` (Section 2). 

Just note that one of the values below doesn't come from `raw.info`: the **data object** (what kind of MNE object the file opened as).

In [ ]:
# Count the channels of each type, e.g. {"mag": 192, "ref_meg": 6, "stim": 20, "misc": 6}
types = Counter(raw.get_channel_types())

# Digitisation points stored in the FIF (fiducials + head shape). "or []" gives an
# empty list instead of None when there are none, e.g. in the empty-room file.
dig = raw.info["dig"] or []

# Each line prints a label padded to 26 characters (":26s") so the values line up.

# The kind of MNE object the file opened as. read_raw_fif always gives a continuous
# Raw object; there is no "recording type" field in a FIF.
print(f"{'data object':26s} {type(raw).__name__}   (continuous recording)")

# Mains frequency in Hz, set by cMEG2FIF (--line-freq). Cerca's JSON had 0.
print(f"{'line_freq (Hz)':26s} {raw.info['line_freq']}")

# Number of channels of each type (0 if that type isn't present)
for t in ["mag", "ref_meg", "stim", "misc"]:
    print(f"{'channels: ' + t:26s} {types.get(t, 0)}")

# Free-text recording comment, copied from Cerca's TaskDescription.
# "!r" shows the raw value, so an empty string appears as '' rather than nothing.
print(f"{'description':26s} {raw.info['description']!r}")

# Count digitisation points by kind: 1 = fiducial (nasion, LPA, RPA), 4 = head-shape point
print(f"{'fiducials':26s} {sum(d['kind'] == 1 for d in dig)}")
print(f"{'head-shape points':26s} {sum(d['kind'] == 4 for d in dig)}")


**What's fixed after running cMEG2FIF, compared with Cerca's `_meg.json` (Section 2):**

- **Line frequency** (`line_freq`) is **60 Hz** instead of `0` (the converter's `--line-freq`, default 60).
- **Recording type:** a FIF has no recording-type field. Instead, the file opens as a continuous `Raw` object, and the future BIDS step writes `RecordingType: "continuous"` for it, replacing Cerca's `"Measurement"` / `"Noise"`.
- **Channel counts:** the 6 reference channels are now their own type (`ref_meg`), so `mag` is 192 rather than 198. The stim channels include the decoded `STI101` and `STI_BTN`, and the button inputs moved from misc to stim.
- **Digitisation:** the task FIF holds the fiducials and head-shape points from the head scan, so `DigitizedLandmarks` and `DigitizedHeadPoints` will be `true`.

**Not fixed:**

- The **descriptive fields** (institution, manufacturer, model, …) are not stored in a FIF. They have to be supplied at the future BIDS step.
- **`description`** is empty because Cerca's `TaskDescription` was empty. Typing a comment into the cAcq software *during the session* would fill both.

## 5. Comparing raw `channels.tsv` to FIF channels

Let's count and compare the channels.

In [ ]:
tsv_names, fif_names = set(ch_tsv["name"]), set(raw.ch_names)
print(f"channels.tsv: {len(tsv_names)}   FIF: {len(fif_names)}")
print("\nIn channels.tsv but not (by that name) in the FIF:")
print("   ", sorted(tsv_names - fif_names, key=lambda s: (len(s), s)))
print("\nIn the FIF but not in channels.tsv:")
print("   ", sorted(fif_names - tsv_names))
print("\nReference channels in the FIF:", sorted(n for n, t in zip(raw.ch_names, raw.get_channel_types())
                                              if t == "ref_meg"))
print("\nThe renamed BNC inputs are explained by the peripherals file. "
      "STI101 and STI_BTN are the two channels the converter creates.")

## 6. Bad channels

Bad channels are stored in `raw.info['bads']`. MNE excludes them from most processing. cMEG2FIF copies any channel marked `bad` in Cerca's `channels.tsv` into this list.

In [ ]:
print("Task recording  info['bads']:", raw.info["bads"])
print("Empty room      info['bads']:", raw_er.info["bads"])
print("\nchannels.tsv 'status' values:")
print(ch_tsv["status"].value_counts().to_string())

If every status is `undetermined`, nobody marked channels during acquisition, so `info['bads']` is empty. That does **not** mean every channel is good. It's your job to look.



### 6.1 Find candidate bad channels

A screen of **both recordings, from start to end**: it flags the OPM channels that are **flat** or **noisy**
compared with the others. These are only *candidates*. Always confirm them in the spectra and traces
(Section 9) before you accept them.

How it works. This is the same screen as Step 7 of the first preprocessing notebook
(`02-preprocessing/01_Explore_Data_and_Sensor_Quality.ipynb`), which explains it in more detail:

- Each recording is cut into segments of `SCREEN_SEGMENT_S` (5) seconds. In each segment, filtered to 1–40 Hz so
  that slow drift and line noise don't count, the screen measures how much each channel's signal varies (its
  **standard deviation**) and compares it with all the other channels in that same segment.
- A channel is **flat** in a segment when its signal is **less than a tenth** of the median channel's
  (`SCREEN_FLAT_RATIO`, 0.1): the sensor, or one of its directions, is off, not locked, or not recording.
- A channel is **noisy** in a segment when it varies far more than the rest: a **robust z-score** above `SCREEN_Z`
  (5). The z-score uses the median and the median absolute deviation instead of the mean and standard deviation,
  so a few very bad channels can't hide themselves by pulling the average towards them.
- A channel is **flagged** when it is flat (or noisy) in at least `SCREEN_MIN_SEGMENTS` (5) segments, so a short
  glitch doesn't count but a sensor that fails partway through does. That is why the whole recording is screened:
  in the sample session, sensor 2O's Y direction stopped working in the last 7 minutes, which a check of the first
  minutes can't see.
- With `FLAT_MARKS_WHOLE_SENSOR = True`, when one direction of a sensor is flat its other two directions are flagged
  too. That is likely best practice, but it is up to you (`False` flags only the flat channel).

Why "flat" uses a ratio and not the z-score: a working channel that is always somewhat quieter than the rest sits at
a z-score of about −3 and, over hundreds of segments, dips below −5 now and then by chance. In the sample session a
real dropout fell to about 1% of the median channel, while a quiet but working channel never went below 12%.

The **empty-room recording** is screened the same way. Notebook 04 marks each bad channel in both recordings, so
both need checking. A channel noisy in both points to the sensor; noisy only with the participant in the helmet
points to the participant (movement, muscle, a loose fit at that spot).

Only the OPM sensors on the head are screened. The reference sensors (3N and 3O, Section 2.1) are left out: they
sit away from the head, so they record no brain signal, heartbeat or head movement, and their signal is naturally
much smaller. Compared with the head sensors, a working reference sensor can look "flat". With only six reference
channels (two sensors, three directions each), comparing them with each other wouldn't be reliable either; check
them by eye in the spectra and traces if you plan to use them (for example for noise reduction).

The cell saves the candidates, plus any channel cMEG2FIF already marked bad, to
`<task FIF name>_bad_channel_candidates.tsv` next to the FIF. Each row says **what kind** of candidate it is
(`flag`: `flat`, `flat sensor` for the other directions of a flat sensor, `noisy`, or `marked bad in Cerca`) and
**in which recording**. Notebook 04 can start from that list (`bad_channels: auto` in the pipeline's subject file),
and its table shows the same labels. Noisy channels are on the list too, but "noisy" has several causes and some
can be fixed by later cleaning, so look at them before you accept them (`keep` in notebook 04 leaves a channel good).

In [ ]:
# ---- Settings for the screen (you can change these) ----------------------------
SCREEN_SEGMENT_S = 5.0          # length of each screened segment, in seconds
SCREEN_Z = 5.0                  # robust z-score above which a channel is "noisy" in a segment
SCREEN_FLAT_RATIO = 0.1         # "flat" in a segment: standard deviation below this fraction of the median channel's
SCREEN_MIN_SEGMENTS = 5         # flagged when flat (or noisy) in at least this many segments
FLAT_MARKS_WHOLE_SENSOR = True  # True: when one direction of a sensor is flat, all three (X, Y, Z) are flagged
(SCREEN_SEGMENT_S, SCREEN_Z, SCREEN_FLAT_RATIO, SCREEN_MIN_SEGMENTS, FLAT_MARKS_WHOLE_SENSOR) = from_config(
    "SCREEN_SEGMENT_S", "SCREEN_Z", "SCREEN_FLAT_RATIO", "SCREEN_MIN_SEGMENTS", "FLAT_MARKS_WHOLE_SENSOR")

# A channel name is the sensor followed by its direction in brackets, e.g. "2C[X]": sensor 2C, direction X
NAME_PATTERN = re.compile(r"^(.*?)\s*\[([XYZ])\]$")


def sensor_of(name):
    # The sensor a channel belongs to, e.g. "2C[X]" -> "2C" (the name itself if it doesn't fit the pattern)
    found = NAME_PATTERN.match(name)
    return found.group(1) if found else name


def screen_segments(r, names, segment_s=5.0):
    # The robust z-score of each channel in each segment (rows: channels, columns: segments), each channel's
    # standard deviation in each segment (tesla), and the start time of each segment (s). Only about a minute is
    # read from disk at a time, so this works on recordings that aren't loaded into memory.
    sfreq = r.info["sfreq"]
    seg_len = int(round(segment_s * sfreq))          # samples per segment
    n_seg = r.n_times // seg_len                     # whole segments only (a short piece at the end is left out)
    pad = int(round(2.0 * sfreq))                    # 2 s of extra data on each side while filtering
    per_block = max(1, int(60 // segment_s))         # filter about a minute at a time, to save memory
    sd = np.zeros((len(names), n_seg))
    for first in range(0, n_seg, per_block):
        last = min(first + per_block, n_seg)
        start, stop = first * seg_len, last * seg_len               # this block's samples
        lo, hi = max(0, start - pad), min(r.n_times, stop + pad)    # ... with the padding
        data = r.get_data(picks=names, start=lo, stop=hi)           # tesla
        data = mne.filter.filter_data(data, sfreq, 1.0, 40.0, verbose=False)   # ignore drift and line noise
        data = data[:, start - lo:stop - lo]                        # drop the padding again
        sd[:, first:last] = data.reshape(len(names), last - first, seg_len).std(axis=2)
    log_sd = np.log10(sd + 1e-30)                    # log scale (the tiny 1e-30 avoids log of 0 for dead channels)
    med = np.median(log_sd, axis=0)                  # the median channel, in each segment
    mad = np.median(np.abs(log_sd - med), axis=0) * 1.4826 + 1e-12   # robust "typical spread", in each segment
    z = (log_sd - med) / mad                         # robust z-score of each channel in each segment
    return z, sd, np.arange(n_seg) * segment_s


def summarize_screen(names, z, sd, seg_times):
    # One row per channel: in how many segments it was flat or noisy, when that first happened, and its flag
    sd_ratio = sd / np.median(sd, axis=0)            # each channel's SD relative to the median channel, per segment
    is_flat = sd_ratio < SCREEN_FLAT_RATIO
    is_noisy = z > SCREEN_Z
    flat_n, noisy_n = is_flat.sum(axis=1), is_noisy.sum(axis=1)
    table = pd.DataFrame({"channel": list(names),
                          "flag": np.where(flat_n >= SCREEN_MIN_SEGMENTS, "flat",
                                           np.where(noisy_n >= SCREEN_MIN_SEGMENTS, "noisy", "")),
                          "flat_segments": flat_n, "noisy_segments": noisy_n, "n_segments": z.shape[1],
                          "median_z": np.median(z, axis=1),                 # the channel's typical z-score
                          "lowest_sd_ratio": sd_ratio.min(axis=1),          # its smallest SD vs the median channel
                          "median_sd_fT": np.median(sd, axis=1) * 1e15})    # its typical standard deviation (fT)
    # When the problem started: the start time of the first flat (or noisy) segment
    first = []
    for flat_row, noisy_row, flag in zip(is_flat, is_noisy, table["flag"]):
        hits = np.flatnonzero(flat_row) if flag == "flat" else np.flatnonzero(noisy_row)
        first.append(seg_times[hits[0]] if flag and len(hits) else np.nan)
    table.insert(2, "first_flagged_s", first)
    # Most affected channels first: the most flagged segments, then the most extreme typical z-score
    table["_most"] = np.maximum(flat_n, noisy_n)
    table["_abs_z"] = table["median_z"].abs()
    return table.sort_values(["_most", "_abs_z"], ascending=False).drop(columns=["_most", "_abs_z"])


def describe(row, recording):
    # The reason saved for a flagged channel, e.g. "flat in 84 of 362 5-s segments of the task recording, ..."
    if row["flag"] == "flat":
        return (f"flat in {row['flat_segments']} of {row['n_segments']} {SCREEN_SEGMENT_S:g}-s segments of the {recording} "
                f"recording, first at {row['first_flagged_s']:.0f} s (standard deviation below "
                f"{100 * SCREEN_FLAT_RATIO:g}% of the median channel's, 1-40 Hz; lowest {100 * row['lowest_sd_ratio']:.1f}%)")
    return (f"noisy in {row['noisy_segments']} of {row['n_segments']} {SCREEN_SEGMENT_S:g}-s segments of the {recording} "
            f"recording, first at {row['first_flagged_s']:.0f} s (robust z above {SCREEN_Z:g}, 1-40 Hz; "
            f"typical z {row['median_z']:+.1f})")


# ---- Screen both recordings ---------------------------------------------------------
# The OPM sensors on the head ("mag"), not the reference sensors ("ref_meg"); already-bad channels included
recordings = {"task": raw, "empty room": raw_er}
screens = {}
for label, r in recordings.items():
    names = [r.ch_names[k] for k in mne.pick_types(r.info, meg="mag", ref_meg=False, exclude=[])]
    z, sd, times = screen_segments(r, names, SCREEN_SEGMENT_S)
    screens[label] = {"names": names, "z": z, "times": times, "table": summarize_screen(names, z, sd, times)}
    table = screens[label]["table"]
    flagged = table[table["flag"] != ""]
    print(f"{label}: {z.shape[1]} segments of {SCREEN_SEGMENT_S:g} s, 1-40 Hz")
    print(f"   flat : {list(flagged.loc[flagged['flag'] == 'flat', 'channel']) or 'none'}")
    print(f"   noisy: {list(flagged.loc[flagged['flag'] == 'noisy', 'channel']) or 'none'}")
    display((flagged if len(flagged) else table.head(10)).round(2))

# ---- Combine into one list of candidates, each with its kind and recording -----------
# One row per channel. If a channel is flagged in several ways, the strongest kind names it
# (flat, then flat sensor, then noisy, then marked bad in Cerca) and the reason lists them all.
RANK = {"flat": 0, "flat sensor": 1, "noisy": 2, "marked bad in Cerca": 3}
found = {}                                    # channel -> {"flag", "recordings", "reasons"}


def add_candidate(channel, flag, recording, reason):
    entry = found.setdefault(channel, {"flag": flag, "recordings": [], "reasons": []})
    if RANK[flag] < RANK[entry["flag"]]:
        entry["flag"] = flag
    if recording and recording not in entry["recordings"]:
        entry["recordings"].append(recording)
    entry["reasons"].append(reason)


for label, s in screens.items():
    for _, row in s["table"][s["table"]["flag"] != ""].iterrows():
        add_candidate(row["channel"], row["flag"], label, describe(row, label))
if FLAT_MARKS_WHOLE_SENSOR:                   # the other directions of each flat sensor
    for label, s in screens.items():
        flat_sensors = {sensor_of(n) for n in s["table"].loc[s["table"]["flag"] == "flat", "channel"]}
        for name in s["names"]:
            if sensor_of(name) in flat_sensors and found.get(name, {}).get("flag") != "flat":
                add_candidate(name, "flat sensor", label,
                              f"another direction of sensor {sensor_of(name)} is flat in the {label} recording")
for label, r in recordings.items():           # channels cMEG2FIF copied as bad from Cerca's channels.tsv
    for ch in r.info["bads"]:
        add_candidate(ch, "marked bad in Cerca", label, f"marked bad in Cerca's channels.tsv ({label} recording; copied by cMEG2FIF)")

cands = pd.DataFrame([{"channel": ch, "flag": e["flag"], "recording": ", ".join(e["recordings"]),
                       "reason": "; ".join(e["reasons"])} for ch, e in found.items()],
                     columns=["channel", "flag", "recording", "reason"])
cands = cands.sort_values(["flag", "channel"], key=lambda col: col.map(RANK) if col.name == "flag" else col)

# ---- Save the candidates next to the task FIF -------------------------------------
# Notebook 04 can use this file (bad_channels: auto in the pipeline's YAML files). It's a suggestion: look at
# the traces and spectra before you accept it.
CANDIDATES_TSV = DATA_FIF.with_name(DATA_FIF.stem + "_bad_channel_candidates.tsv")
cands.to_csv(CANDIDATES_TSV, sep="\t", index=False)
print(f"\n{len(cands)} candidate(s), saved to {CANDIDATES_TSV}:")
display(cands if len(cands) else "none")

#### When each channel was flat or noisy

A map of each screen: one row per OPM channel (in the file's order, so the three directions of a sensor are next to
each other), one column per segment. The colour is the channel's robust z-score in that segment: white is typical,
**red** is louder than the others (above +5 counts as noisy), **blue** is quieter; values beyond ±10 are drawn as
±10. The names of flagged channels are in bold, red for noisy and blue for flat.

What to look for:

- a row that turns **blue partway and stays blue**: a sensor that stopped working during the recording;
- a row that is **red all along**: a noisy sensor;
- **short red patches** on a few neighbouring sensors: often muscle or movement under those sensors.

Each segment is compared with the median channel of that same segment, so something that affects **all** sensors at
once (for example a large head movement) doesn't show up here. Look for that in the traces (Section 9.2).

In [ ]:
def plot_screen_map(z, seg_times, names, table, title):
    # The screen as a picture: rows are channels, columns are segments, colour is the robust z-score
    fig, ax = plt.subplots(figsize=(14, max(4.0, 0.065 * len(names) + 1.5)))
    image = ax.imshow(np.clip(z, -10, 10), aspect="auto", cmap="RdBu_r", vmin=-10, vmax=10, interpolation="nearest",
                      extent=[seg_times[0] / 60, (seg_times[-1] + SCREEN_SEGMENT_S) / 60, len(names) - 0.5, -0.5])
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=4)
    flag_of = dict(zip(table["channel"], table["flag"]))
    for label in ax.get_yticklabels():          # flagged channels: bold, red (noisy) or blue (flat)
        flag = flag_of.get(label.get_text(), "")
        if flag:
            label.set_color("tab:red" if flag == "noisy" else "tab:blue")
            label.set_fontweight("bold")
    ax.set_xlabel("Time (min)")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label="robust z-score (beyond ±10 drawn as ±10)", fraction=0.02, pad=0.01)
    fig.tight_layout()
    plt.show()


for label, s in screens.items():
    plot_screen_map(s["z"], s["times"], s["names"], s["table"],
                    f"Flat/noisy screen, {SCREEN_SEGMENT_S:g}-s segments: {label} recording")

To mark channels as bad for your analysis (in memory; this doesn't change the file on disk):

```python
raw.info["bads"] = ["2A[X]", "2A[Y]"]         # example names
```

On OPMs a problem usually affects a whole sensor, so check all three axes of a suspect sensor. You can also mark bads interactively by clicking traces in `raw.plot()`.

## 7. The conversion log

Every run writes `<output>_conversion_log.txt` next to the FIF. It's the **provenance record**. Keep it with the data.

This cell just prints out the most important lines.

In [ ]:
def parse_log(text):
    info = {}
    for key, pat in [("version", r"cMEG2fif version (\S+)"), ("run", r"Run:\s+(.+)"),
                     ("host", r"Host:\s+(.+)"), ("versions", r"Versions:\s+(.+)"),
                     ("output", r"Output:\s+(.+)"), ("peripherals", r"Peripherals:\s+(.+)"),
                     ("helmet", r"Helmet:\s+(.+)"), ("start", r"Recording start.*?:\s+(.+)")]:
        m = re.search(pat, text)
        info[key] = m.group(1).strip() if m else None
    info["summary"] = next((l.strip() for l in text.splitlines() if "MEG channels," in l), None)
    info["used_lines"] = [l.strip() for l in text.splitlines()
                          if re.match(r"\s*bit\s+\d+", l) and "unused" not in l]
    info["unused_lines"] = [l.strip() for l in text.splitlines()
                            if re.match(r"\s*bit\s+\d+", l) and "unused" in l]
    info["warnings"] = [l.strip() for l in text.splitlines()
                        if re.search(r"warn|error|mismatch|jump|incomplete", l, re.I)]
    info["bads"] = [l.strip() for l in text.splitlines() if re.search(r"\bbad", l, re.I)]
    return info

def log_path_for(fif):
    # sub-01_..._meg.fif -> sub-01_..._meg_conversion_log.txt (written next to the FIF)
    return fif.with_name(fif.stem + "_conversion_log.txt")

log_text = log_path_for(DATA_FIF).read_text()
log = parse_log(log_text)
for k in ["version", "run", "host", "versions", "output", "helmet", "peripherals", "start", "summary"]:
    print(f"{k:12s} {log[k]}")
print("\nLines that carried a signal:")
for l in log["used_lines"]:
    print("   ", l)
print("\nLines that never changed (unused):")
for l in log["unused_lines"]:
    print("   ", l)
print("\nWarnings / problems:", log["warnings"] or "none")
print("Bad-channel lines:  ", log["bads"] or "none")

**Questions to answer from the log:**

1. Which VPixx trigger bits were used? What's the largest code `STI101` could contain?
2. Which response buttons did the participant actually press?

## 8. Peripherals, triggers and buttons

### 8.1 The peripherals file

Cerca labels its analog inputs only as `BNC 1` … `BNC 16`. The **peripherals file** (`cMEG_peripherals.tsv`) says what is plugged into each one, what the converter should name it, and what the converter should **do** with it. Its `type` column is an instruction to cMEG2FIF, not an MNE channel type. At our facility it has two values:

| `type` in the peripherals file | What cMEG2FIF does | MNE channel type in the FIF |
|---|---|---|
| `button` | keeps the line **and** adds it as a bit to `STI_BTN` (first button = 1, then 2, 4, 8, …) | `stim` |
| `misc` | keeps the voltage as recorded (e.g. the eye tracker's analog output) | `misc` |

MNE has no "button" channel type, so button inputs become `stim` channels. The three eye tracking channels and the remaining 3 BNC inputs that are not listed in the file are kept as `misc`.

The cell below loads the peripherals file the converter used (its path is in the conversion log).

In [ ]:
periph_path = Path(log["peripherals"])      # from the conversion log (Section 7)
print("Peripherals file used by the converter:", periph_path)

### 8.2 Buttons → `STI_BTN`

Each button line becomes one bit of `STI_BTN`, in the order listed in the peripherals file: `R_thumb`=1, `R_index`=2, `R_middle`=4, … `L_pinky`=512. The value tells you **which** button was pressed. If two buttons are held at once, their bits add (e.g. `R_index` + `L_index` = 2 + 64 = 66).

In [ ]:
# Load the peripherals file (path from Section 8.1); its "button" rows set the STI_BTN bits
periph = read_tsv(periph_path)

trig_names = [c for c in raw.ch_names if c.startswith("Trigger")]
btn_bits = {}
if periph is not None:
    btn_names = list(periph.loc[periph["type"].str.lower() == "button", "name"])
    btn_bits = {n: 2 ** i for i, n in enumerate(btn_names)}
btn_names = [n for n in btn_bits if n in raw.ch_names]
analog_names = [c for c, t in zip(raw.ch_names, raw.get_channel_types()) if t == "misc"]
print("Trigger lines :", trig_names)
print("Button lines  :", btn_names)
print("Analog (misc) :", analog_names)

events = mne.find_events(raw, stim_channel="STI101", consecutive=True, verbose=False)
buttons = mne.find_events(raw, stim_channel="STI_BTN", verbose=False)
print(f"\nSTI101: {len(events)} events   STI_BTN: {len(buttons)} presses")

def plot_window(r, picks, center_sample, pre=0.5, post=8.5, combined=None, title=""):
    # Plot a few seconds of the `picks` channels around `center_sample`,
    # with the decoded code channel (`combined`) as a step plot underneath.
    sf = r.info["sfreq"]
    start = max(int(center_sample - r.first_samp - pre * sf), 0)
    stop = min(int(center_sample - r.first_samp + post * sf), r.n_times)
    data, times = r.get_data(picks=picks, start=start, stop=stop, return_times=True)
    n = len(picks) + (1 if combined else 0)
    fig, axes = plt.subplots(n, 1, figsize=(10, 0.55 * n + 1.2), sharex=True)
    axes = np.atleast_1d(axes)
    # Same y-range for every line, so a line that never switches stays flat
    # instead of being stretched to fill its panel
    lo, hi = data.min(), data.max()
    pad = 0.05 * (hi - lo)
    for ax, name, y in zip(axes, picks, data):
        ax.plot(times, y, lw=0.8)
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=8)
        ax.set_yticks([])
    if combined:
        c, t = r.get_data(picks=[combined], start=start, stop=stop, return_times=True)
        axes[-1].step(t, c[0], where="post", color="k", lw=1.2)
        axes[-1].set_ylabel(combined, rotation=0, ha="right", va="center", fontweight="bold")
        for val in np.unique(c[0][c[0] > 0])[:10]:
            axes[-1].text(t[np.argmax(c[0] == val)], val, f" {int(val)}", fontsize=8, va="bottom")
    axes[-1].set_xlabel("time (s)")
    fig.suptitle(f"{title}\n(all lines on the same scale: {lo:.1f} to {hi:.1f} V)")
    fig.tight_layout()
    plt.show()

if len(buttons):
    plot_window(raw, btn_names, buttons[0, 0], pre=1.0, post=120.0, combined="STI_BTN",
                title="Button lines (V) and the decoded STI_BTN code")
else:
    print("No button presses in this recording.")

### 8.3 Analog eye channels

`eye_x`, `eye_y` and `eye_pupil` carry the EyeLink's analog output (in volts). BNC 14–16 are free inputs, so they should be flat. (The EyeLink `.EDF` file itself will be covered later)

In [ ]:
start = int(min(3000 * raw.info["sfreq"], raw.n_times // 2))
stop = min(start + int(20 * raw.info["sfreq"]), raw.n_times)
data, times = raw.get_data(picks=analog_names, start=start, stop=stop, return_times=True)
fig, axes = plt.subplots(len(analog_names), 1, figsize=(10, 1.1 * len(analog_names) + 1), sharex=True)
for ax, name, y in zip(np.atleast_1d(axes), analog_names, data):
    ax.plot(times, y, lw=0.6)
    ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=8)
    ax.text(1.0, 0.5, f" range {np.ptp(y):.3f} V", transform=ax.transAxes, fontsize=8, va="center")
np.atleast_1d(axes)[-1].set_xlabel("time (s)")
fig.suptitle("Analog BNC inputs, 20 s of the task recording")
fig.tight_layout()
plt.show()

### 8.4 VPIXX Triggers → `STI101`

The 8 VPixx trigger lines are recorded as voltages. Each line is one **bit** of the stimulus code. The converter thresholds each line and adds them up: `STI101 = 1·T1 + 2·T2 + 4·T3 + … + 128·T8`.

The plot shows a few seconds around the first stimulus code: the raw trigger voltages on top, the combined `STI101` code underneath.

In [ ]:
if len(events):
    plot_window(raw, trig_names, events[0, 0], combined="STI101",
                title="VPixx trigger lines (V) and the decoded STI101 code")

## 9. A first look at signal quality

### 9.1 Spectra: empty room vs. task

The empty room shows the environment's noise floor with nobody present. Compare it with the task recording. Look for:

- **60 Hz and harmonics** (120, 180 Hz): mains interference
- **Low-frequency rise** below a few Hz: drift, movement, residual fields
- **Alpha (~10 Hz)** in the task data but not the empty room: brain signal

Each spectrum uses one minute of data (only that minute is read from disk).

In [ ]:
def mean_psd(r, tmin=30.0, dur=60.0, fmax=200.0):
    tmax = min(tmin + dur, r.times[-1])
    tmin = max(0.0, tmax - dur)
    seg = r.copy().crop(tmin, tmax).pick("mag", exclude="bads").load_data()
    psd = seg.compute_psd(fmin=0.5, fmax=fmax, n_fft=int(4 * seg.info["sfreq"]), verbose=False)
    amp = np.sqrt(psd.get_data()) * 1e15                         # fT / sqrt(Hz)
    return psd.freqs, np.median(amp, axis=0), np.percentile(amp, [10, 90], axis=0)

plt.figure(figsize=(10, 4.5))
for label, r, c in [("empty room", raw_er, "tab:gray"), ("task", raw, "tab:blue")]:
    f, med, (lo, hi) = mean_psd(r)
    plt.semilogy(f, med, color=c, label=f"{label} (median over sensors)")
    plt.fill_between(f, lo, hi, color=c, alpha=0.15)
for h in (60, 120, 180):
    plt.axvline(h, color="r", ls=":", lw=0.8)
plt.xlabel("frequency (Hz)"); plt.ylabel("amplitude (fT/√Hz)")
plt.title("OPM noise spectra (shaded: 10th–90th percentile across sensors)")
plt.legend(); plt.tight_layout(); plt.show()

### 9.2 Browse the raw data

`raw.plot()` is MNE's data browser. `%matplotlib widget` (from `ipympl`, installed in Section 0) makes it interactive inside the notebook, which works both on scotty and on your laptop. Click inside the figure first so it has keyboard focus, then:

- **← / →** scroll in time
- **↑ / ↓** scroll through channels
- **Click a trace** to mark that channel bad (written to `raw.info["bads"]` when you close the figure with the × in its toolbar)

On scotty, every redraw travels over the tunnel, so keep `duration` and `n_channels` small if it feels sluggish.

`BROWSE_STARTS` lists where each window starts. When the pipeline runs this notebook, each window is
saved as a still picture instead.

In [ ]:
# Run by hand: an interactive browser (needs ipympl; "%matplotlib widget" does the same).
# Pipeline mode: still pictures of the same window(s), since there is no screen to interact with.
get_ipython().run_line_magic("matplotlib", "widget" if PIPELINE is None else "inline")
import mne

BROWSE_STARTS = [60]   # start of each 10-s window to show, in seconds; e.g. [60, 1800] for two windows
BROWSE_STARTS = from_config("BROWSE_STARTS")

mne.viz.set_browser_backend("matplotlib")
for start in BROWSE_STARTS:
    raw.plot(start=start, duration=10, n_channels=30, events=events,  # raw = the task recording, opened in Section 4.
             picks=["mag", "stim"], scalings=dict(mag=2e-11, stim=1e3), show=True);

## 10. Summary

- **§2, §4 — Sidecar:** Cerca's `_meg.json` is not valid BIDS (`PowerLineFrequency: 0`, `RecordingType: "Measurement"`, empty descriptive fields, wrong channel counts). cMEG2FIF doesn't write a sidecar, but it puts the correct values into the FIF, and the future BIDS step will write the sidecar from there.
- **§5 — Channel types:** the FIF has four: `mag`, `ref_meg`, `stim`, `misc`.
- **§6 — Bad channels:** `info['bads']` starts empty unless channels were marked in Cerca; finding bad channels is your job.
- **§7 — Conversion log:** records provenance and every decision the converter made.
- **§8 — Peripherals:** the peripherals file maps BNC inputs to named button and analog channels; `STI101` holds VPixx codes and `STI_BTN` holds button identity as bits.